# 第26章 卒業制作と、2030年への地図

自分の問いを立て、実験し、誠実に報告する。そして、その先へ

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：自分で決めたテーマの卒業制作（問い・実験・結果・考察・公開）と、2030年の卒業までの学びの計画
- **身につく数学・理論**：研究の進め方（問い→仮説→実験計画→記録→結論） ／ 再現性（シード・版・設定の記録） ／ 論文の読み方（3回読み）と、読む順番 ／ AIを作る人の倫理
- **新しく使うPython・PyTorch**：`importlib.metadata` で道具の版を記録する ／ `json` で実験の設定を保存する

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Colab・ローカルどちらでも |
| 所要時間目安 | 数週間（自分のペースで） |
| 難易度 | ★★★★☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> ここまでの第0〜25章（全26章）で、あなたはLLMを動かし、ゼロから作り、鍛え、測り、システムにして、公開しました。最後の章では、**自分で問いを立てて、自分の答えを出す**卒業制作に取り組みます。そして、2030年の卒業までの4年間を、どう使うかの地図を描きます。

> **🔗 これまでの復習とつながり：第1部〜第7部の総まとめ：第1〜25章の知識がつながる地図**
>
> ここまで積み上げてきた7つの部は、すべて1本の線でつながっています。卒業制作では、自分の「問い」に合わせてこの道具箱から必要な部品を自由に組み合わせます。
>
> - **第1部（推論の仕組み）**：トークンID → ロジット → 温度つき softmax → 自己回帰生成
> - **第2部（学習の数学と実装）**：対数と交差エントロピー → 自動微分と勾配降下法 → LayerNorm と Adam
> - **第3部（Transformer自作）**：内積とQKV Attention → 残差接続・ブロック → 夏目漱石GPTの事前学習
> - **第4部（大規模化と高速化）**：BPE → GPT-2重み読み込み → スケーリング則と **KVキャッシュ**
> - **第5部（事後学習と評価）**：SFT → 量子化と **LoRA / QLoRA** → 信頼区間・LLM審査員による評価（第20章）
> - **第6部（応用と公開）**：RAG → RLHF / **DPO** → ツールを使うエージェント → ブラウザ静的公開
> - **第7部（研究への入口）**：RMSNorm・SwiGLU・RoPE・MoE・GRPO と、公平な比較・アブレーション実験

**ボンゴ先生**：キミが最初の章で書いたのは `print("こんにちは、LLM！")` だった。今のキミは、Attentionを一行ずつ書けて、GPT-2の重みを読み込めて、DPOの損失を式から実装できて、それをWebページで世界に公開できる。半年前のキミに見せたら、たぶん信じないだろうね。

**ミナト**：正直、自分でも信じられません。でも、まだ分からないところもたくさんあります。

**ボンゴ先生**：それでいい。研究者というのは、分からないことの一覧を、一番たくさん持っている人のことだ。その一覧こそが、キミの卒業制作のテーマの宝箱なんだよ。

## 振り返り：あなたが作ったもの

| 部 | 作ったもの | 身についた理論 |
|---|---|---|
| 第1部 | LLMとの会話、トークン観察器、自作の生成ループ | 次トークン予測、UTF-8、softmax、温度 |
| 第2部 | bigram、損失計算、ニューラルbigram、MLP、学習の診察表 | 条件付き確率、対数、行列、微分、勾配降下法、正規化、Adam |
| 第3部 | Attention、Transformer、漱石GPT | 内積、QKV、残差接続、混合精度、データの量 |
| 第4部 | BPE、GPT-2の読み込み、KVキャッシュ | 圧縮、重み共有、スケーリング則、6ND、分散学習 |
| 第5部 | ボンゴ先生SFT・LoRA・評価レポート | 損失マスク、低ランク、量子化、ブートストラップ |
| 第6部 | RAG、DPO、エージェント、公開したWebページ | コサイン類似度、ブラッドリー・テリー、ツール呼び出し |
| 第7部 | 現代版ミニGPT、比較実験 | RoPE、RMSNorm、SwiGLU、MoE、アブレーション |

#### 📝 確認問題（`ch26-q01` · Lv.2 · 【第1〜7部総合】LLM開発の課題と最適な技術・章の対応づけ）

LLMシステムを開発中に次の3つの課題にぶつかりました。最も適切な解決アプローチの組み合わせはどれですか？
① 長い文章を生成するほど1トークンあたりの待ち時間が急激に遅くなる
② 無料GPU（VRAM 15GB）で大きめのモデルを微調整すると `CUDA out of memory` が出る
③ モデルが学校の最新行事について平気で嘘（ハルシネーション）をつく

- **a.** ① ドロップアウト率を上げる（第12章）、② 生成時の温度を2.0へ上げる（第3章）、③ 語彙サイズを10へ減らす（第4章）
- **b.** ①〜③のすべての課題に対して、学習率 `learning_rate` を10倍へ引き上げて再学習すれば一括で根本解決できる
- **c.** ① KVキャッシュ（第16章）、② 量子化＋LoRA／QLoRA（第19章）、③ RAG（第21章）または検索つきエージェント（第23章）
- **d.** ① DPOで好み学習を行う（第22章）、② BPEの結合回数を増やす（第4章）、③ モデルをbase64形式へ変換する（第24章）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：3つの課題をそれぞれ「推論の重複計算（第4部）」「メモリ節約の微調整（第5部）」「外部知識の参照（第6部）」に結びつけましょう。
- **ヒント 2**：過去のK/V再利用、4ビット化＋低ランク付せん学習、外部資料検索の3技術の組み合わせを探してください。
- **正解**：`c`

**考え方**：第1〜7部で学んだ技術はバラバラの暗記項目ではなく、「速度・メモリ・知識・性格・推論・公開」という具体的な課題に対応する道具箱になっています。

1. 生成速度の低下 → 過去のK/Vを再計算している無駄を省く **KVキャッシュ**（第16章）。
1. 微調整時のVRAM不足 → 重みを4ビット化し付せんだけ学ぶ **QLoRA**（第19章）。
1. 最新・ローカル事実の誤答 → 外部資料を検索して渡す **RAG / エージェント**（第21・23章）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q01

</details>

## 卒業制作のテーマを選ぶ

卒業制作は「作品」と「研究」の両方の性格を持つものがおすすめです。つまり、**何かを作り**、それが**どれくらい良いかを測る**。テーマの例を挙げます。気に入ったものを選んでも、組み合わせても、まったく別のものでもかまいません。

| テーマの例 | 使う章 | 計算の重さ | 問いの例 |
|---|---|---|---|
| A. 別の作家のミニGPT（宮沢賢治、太宰治…） | 4〜13, 24 | 軽い（T4で数分） | 作家ごとの「らしさ」は、モデルの損失でどれくらい区別できる？ |
| B. ○○風LoRA（方言、キャラクター） | 18〜20 | 中くらい | 何件のデータで、口調はどれくらい変わる？ 知識は忘れる？ |
| C. 授業ノートに質問できるRAG | 21, 20 | 軽い | チャンクの大きさで、検索の正答率はどう変わる？ |
| D. 小さな部品のアブレーション | 12, 14, 25 | 軽い〜中 | BPEの語彙の大きさで、1文字あたりの損失はどう変わる？ |
| E. 基本情報の勉強エージェント | 21〜23 | 中くらい | 道具の説明の書き方で、エージェントの成功率はどう変わる？ |
| F. LLM審査員の偏りを測る | 20 | 中くらい | 小さなLLMの審査員には、どんな偏りがある？ |

> **✨ コツ：テーマ選びの3つのコツ**
>
> 1. **小さく**：最初の実験が1時間以内に回るくらいの大きさにする。大きくするのは後でいい
> 2. **測れる**：「良くなった」をどう測るかを、最初に決める（第20章）
> 3. **好き**：自分が面白いと思えるもの。最後まで続ける力は、結局これ

<details><summary><b>基礎の補足：反証可能な仮説（Falsifiable Hypothesis）の立て方</b>（クリックで開く）</summary>

科学的な仮説の条件は、**「もし間違っていたら、実験データによってはっきり『違った！』と否定（反証）できること」**です。

- × **悪い例（曖昧な願望）**：「いろいろ工夫してRAGをいい感じに賢くする」→ どんな結果が出ても「賢くなった気がする」と言えてしまい検証できない。
- ○ **良い例（反証可能な仮説）**：「授業ノートRAGでチャンク長を100文字から300文字に変えると、独立したテスト質問30問における正解資料の検索ヒット率（Hit@2；正解資料が1件のとき Recall@2 と同じ）が10ポイント以上上がる（理由：文脈の途中で途切れにくくなるため）」→ 結果が+3ポイントや低下なら明確に仮説が棄却され、次の深い考察へ進めます！

</details>

#### 📝 確認問題（`ch26-q02` · Lv.3 · 反証可能な仮説（Falsifiable Hypothesis）と曖昧な願望の見分け方）

卒業制作の仮説として、実験で「正しいか間違っているかを客観的に判定できる（反証可能な）」最も良い仮説はどれですか？

- **a.** 最新の論文で提案された現代版GPTアーキテクチャは設計が優れているため、どんな条件でも絶対に最強の性能を発揮する
- **b.** プロンプトやデータ前処理にいろいろな工夫を積み重ねれば、自作のAIチャットボットはきっといい感じに賢くなると思う
- **c.** 授業ノートRAGのチャンク長を100→300文字にすると、テスト30問のRecall@2が10点以上上がる
- **d.** ディープラーニングの計算量がこのまま指数的に増え続ければ、AIはいずれ人間の総合的な知能を超えるかもしれない

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：科学的に良い（反証可能な）仮説には「①何を変えるか」「②何の指標で測るか」「③実験で否定され得る具体的な予想」の3要素が揃っています。
- **ヒント 2**：結果が出たときに「予想通りだった／予想が外れた」を数値で客観的に判定できる文を選んでください。
- **正解**：`c`

**考え方**：科学的な仮説の条件は「間違っていたときに実験データによってはっきり『違った！』と分かること（反証可能性）」です。外れたときにこそ新しい発見が生まれます。

1. 変える条件：チャンク長（100文字 → 300文字）。
1. 測る対象と指標：テスト質問30問での Recall@2。
1. 反証可能性：向上幅が10ポイント未満（あるいは低下）なら仮説は明確に棄却され、原因分析へ進める。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q02

</details>

#### 📝 確認問題（`ch26-q03` · Lv.3 · 卒業制作のテーマ設計（小さく・測れる・ベースラインあり））

卒業制作で「方言キャラ風LoRA」を作ることにしました。最初の一歩の計画として最も現実的で研究価値が高いものはどれですか？

- **a.** LoRAの訓練に使った10個の質問とまったく同じ質問をそのままモデルへ聞いて、「100%完璧に方言で答えられた！」と結論づける
- **b.** 学習データを1件だけ作って短時間で微調整し、生成を何度も試して一番かわいく見えた1個の返事だけをスクリーンショットして終了する
- **c.** 無料Colab上で70Bパラメータの巨大モデルをゼロから事前学習する壮大な計画だけ先に立て、評価指標はモデル完成後にゆっくり考える
- **d.** 1時間以内で回る0.5B＋LoRAでデータ件数を `10/30/100件` と変え、別テストで方言らしさと元知識維持率を学習前と比べて測る

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：卒業制作のテーマ設計のコツ「小さく回せる・客観的に測れる・ベースラインがある」を思い出しましょう。
- **ヒント 2**：1変数を動かして比較すること、未知テストで測ること、さらに「狙った能力（方言）と副作用（知識忘却）」の両面を見る点に注目してください。
- **正解**：`d`

**考え方**：「データ件数という1変数を動かし、狙った能力（方言口調）と副作用（知識の忘却）の両面をベースラインと比較する」設計は、小さく回せてしかも深い考察が書ける最高の卒業制作テーマです。

1. 規模：1時間以内に1実験が終わる0.5B＋LoRAで始める。
1. 変数：データ件数（10 / 30 / 100件）を1つだけ動かす。
1. 評価：学習前ベースラインと比較し、未知テスト質問で「方言らしさ」と「知識維持（忘却の有無）」の2軸を測る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q03

</details>

## 研究の進め方

> **💡 理論のポイント：研究の5ステップ**
>
> 1. **問い**：「〜は〜にどう影響するか？」の形で、1文で書く
> 2. **仮説**：結果を予想して、理由と一緒に書いておく（後で予想と比べるのが大事。第0章の「予想→実行→比べる」）
> 3. **実験計画**：比べる相手（ベースライン）、変える条件、測る指標、シードの数を決める。**変える条件は必ず1つずつ**にします。一度に2つ以上変えてしまうと（例：学習率を上げて同時にモデルの層数も変える）、良くなった原因がどちらにあるのか特定できなくなるからです（対照実験・アブレーションの鉄則）
> 4. **実験と記録**：設定と結果を、全部記録する。うまくいかなかった実験も消さない
> 5. **結論**：言えることと、言えないことを分けて書く（第20章の評価レポート）

<details><summary><b>基礎の補足：実験の4大落とし穴チェック（変数統制・データリーク・ベースライン・統計）</b>（クリックで開く）</summary>

1. **変数の交絡を防ぐ**：一度に2つ以上変えない（例：新部品＋層数増＋ステップ増を同時に変えると原因不明になる）。必ず **1変更＝1実験**（アブレーション）にする。
2. **データリークをゼロにする**：訓練とテストに同一データ・重複窓を入れないのはもちろん、**テストデータを見ながらプロンプトや設定を何度も書き直すのもリーク（テスト過学習）**。調整は検証データだけで行い、テストデータは最終報告の1回だけ使う。
3. **誠実なベースラインを選ぶ**：わざと壊した弱い相手ではなく、素のモデルや単純な既存法（2-gram検索、GPT-2型等）と条件を揃えて比べる。
4. **ばらつきを測る（第20章）**：1回の好結果だけを選ばず、複数シードの幅やブートストラップ95%信頼区間を添え、「差がノイズより大きいか」を判定する。

</details>

実験の設定と、使った道具の版を、毎回ファイルに残しておく習慣をつけましょう。半年後の自分（または、あなたの結果を確かめたい誰か）が、同じ実験をくり返せるようにするためです。これを **再現性** と言います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import json, platform, datetime
from importlib.metadata import version

record = {
    "date": datetime.date.today().isoformat(),
    "question": "BPEの語彙の大きさで、1文字あたりの損失はどう変わるか",
    "config": {"vocab_size": 2000, "n_layer": 2, "n_embd": 64, "steps": 1500, "lr": 3e-3, "seeds": [0, 1, 2]},
    "versions": {"python": platform.python_version(), "torch": version("torch"), "transformers": version("transformers")},
    "results": {},
}
with open("experiment_001.json", "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)
print(json.dumps(record["versions"], ensure_ascii=False))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
{"python": "3.12.11", "torch": "2.9.0+cu126", "transformers": "5.2.0"}
```

</details>

### 報告書のひな形

卒業制作の報告書は、次の形でまとめると、読む人に伝わりやすくなります。論文と同じ構成です。

```text
# タイトル（何をして、何が分かったかが伝わる一文）

## 1. はじめに
- 背景：なぜこの問いが面白いのか
- 問い：〜は〜にどう影響するか？
- この研究で分かったこと（先に結論を1〜3行で）

## 2. 方法
- データ：何を、どれだけ、どう分けたか（訓練・検証・テスト）
- モデル：大きさ、設定、元にしたモデルとライセンス
- 比べる条件と、ベースライン
- 評価の方法：指標、シードの数、信頼区間の出し方

## 3. 結果
- 表とグラフ（平均と、ばらつき・信頼区間）
- 生成例（良い例だけでなく、失敗例も）

## 4. 考察
- 結果から言えること、言えないこと
- 予想と違った点と、その理由の推測
- 限界（データが少ない、モデルが小さい、など）と、次にやりたいこと

## 5. 公開物
- コード（GitHub）、モデル、デモのURL
```

**ボンゴ先生**：「失敗例も載せる」「限界も書く」。ここが一番大事だ。うまくいった例だけを並べるのは、宣伝であって研究じゃない。ファインマンはこれを「積み荷信仰の科学（カーゴ・カルト・サイエンス）」と呼んで戒めた。形だけ科学っぽくても、自分に都合の悪い結果を隠したら、それは科学じゃないとね。

#### 📝 確認問題（`ch26-q04` · Lv.3 · 変数統制（1つずつ変える鉄則）と交絡因子の見抜き方）

学生Aは「新しい学習率スケジュール（コサイン減衰）の効果」を確かめるため、ベースライン（定数学習率・2層・1000ステップ）と提案手法（コサイン減衰・4層・3000ステップ）を比べ、検証損失が `0.4` 下がったので「コサイン減衰は大成功！」と報告しました。この実験の致命的な欠陥はどれですか？

- **a.** コサイン減衰スケジュールは理論上100層以上の巨大モデルでしか使えない仕様なのに、わずか4層で実験してしまったこと
- **b.** 検証損失が `0.4` も大きく下がっている事実だけで十分に価値があるため、実験計画や主張に致命的な欠陥はまったくない
- **c.** 調べたい学習率スケジュールと同時に層数とステップ数も変えており（変数の交絡）、損失低下の原因を一切切り分けられない
- **d.** 学習率スケジュールの良さを測るには検証損失ではなく、保存された重みファイルのバイトサイズだけを比較すべきだったこと

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：対照実験の鉄則「調べたい条件（独立変数）は1つだけ変え、それ以外はすべて揃える」と照らし合わせましょう。
- **ヒント 2**：ベースライン（2層・1000ステップ）と提案手法（4層・3000ステップ）の間で、学習率スケジュール以外に何が同時に変わっているか確認してください。
- **正解**：`c`

**考え方**：調べたい変数（独立変数）以外の条件（層数・ステップ数・データ・シード）を完全に揃えて初めて、結果の差をその変数の効果として因果的に主張できます。

1. 調べたい変数：学習率スケジュール（定数 vs コサイン減衰）。
1. 交絡してしまった変数：層数（2→4）とステップ数（1000→3000）。
1. 修正法：どちらも「2層・1000ステップ（あるいは両方4層・3000ステップ）」に揃えて比較し直す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q04

</details>

#### 📝 確認問題（`ch26-q05` · Lv.3 · データリーク（訓練・検証・テストの汚染）の見抜き方と防止策）

次のうち、**データリーク（評価の汚染）**が起きてしまい、モデルの本当の実力を過大評価してしまう行動はどれですか？

- **a.** 訓練データとテストデータのID集合の共通部分 `set(train) & set(test)` が空であることを毎回コード上の `assert` で自動検証する
- **b.** テスト30問と模範解答を眺めながら満点になるまでプロンプトや設定を20回書き直し、「未知データで正答率100%」と報告する
- **c.** 最初にデータを「訓練・検証・テスト」へ重複なく分割し、調整は検証データだけで行いテストデータは最終報告の1回だけ使う
- **d.** 学習前ベースラインと学習後モデルの両方を、調整に一度も使っていないまったく同じ独立テストセットで同条件のまま評価する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：データリークには「訓練データにテスト問題が混ざる」だけでなく、「テスト問題を見ながら人間が設定を調整し続ける（テストセットへの過学習）」も含まれます。
- **ヒント 2**：テストデータは最終報告の1回まで封印すべきだという原則に反している行動を選びましょう。
- **正解**：`b`

**考え方**：データリークには「訓練データにテスト問題が直接混ざる（直接リーク）」だけでなく、「テスト結果を見ながら何度も設定やプロンプトをいじり続ける（テストセットへの過学習）」も含まれます。調整は必ず検証データで行い、テストデータは最後まで封印します。

1. 訓練データ：モデルの重み学習に使う。
1. 検証データ（val）：学習率・チャンク長・プロンプトなどの試行錯誤に使う。
1. テストデータ（test）：最終方式が確定したあと、真の汎化性能を測るためだけに使う（絶対に調整材料にしない）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q05

</details>

*(この節の追加問題 `ch26-q06`, `ch26-q07` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html で自動採点・復習できます)*

## 論文を読む

卒業制作でも、その先でも、**論文**はあなたの最強の教科書になります。最新の研究のほとんどは、英語の論文として **arXiv**（アーカイブ、<https://arxiv.org/>）で無料で公開されています。

> **💡 理論のポイント：3回に分けて読む**
>
> 論文を最初から最後まで一度に読もうとすると、たいてい挫折します。3回に分けて読みましょう。
>
> 1. **1回目（10分）**：タイトル・要旨（Abstract）・図と表・結論だけ。「何を主張している論文か」をつかむ
> 2. **2回目（1時間）**：本文を通して読む。数式は飛ばしてもいい。分からない用語に印をつける
> 3. **3回目（数時間〜）**：大事な論文だけ。数式を追い、できれば**コードを動かして再現**する
>
> 英語が苦手なら、翻訳ツールやLLMに要約を手伝ってもらうのもいいでしょう。ただし、要約が間違っていることもあるので（第21章のハルシネーション）、大事なところは原文の図や数式で確かめます。

<details><summary><b>基礎の補足：論文を批判的に読むコツ：「主張（Claim）」と「証拠（Evidence）」を切り分ける</b>（クリックで開く）</summary>

タイトルや要旨（Abstract）には著者の最も強い**主張（Claim）**が書かれます。それを鵜呑みにせず、実験セクションで表や条件（**Evidence**）と突き合わせるクセをつけましょう。

- **ベースラインは対等か？**：提案手法だけパラメータが多い・推論計算量が多い・チューニングが手厚いことはないか？
- **アブレーションはあるか？**：複数の工夫のうち、どれが本当に効いているのか切り分けられているか？
- **ばらつきと限界はあるか？**：1シードだけの僅差ではないか？ どんな条件だと失敗するか書かれているか？

</details>

この教科書の各章は、次の論文とつながっています。読む順番の目安として、上から順におすすめします。

| 論文（発表年） | 内容 | 一次リンク | 関係する章 |
|---|---|---|---|
| Attention Is All You Need（2017） | Transformerの提案 | [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) | 11〜12 |
| Language Models are Unsupervised Multitask Learners（2019, GPT-2） | 大規模な事前学習 | [OpenAI PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) | 13, 15 |
| Scaling Laws for Neural Language Models（2020） | スケーリング則 | [arXiv:2001.08361](https://arxiv.org/abs/2001.08361) | 15 |
| LoRA: Low-Rank Adaptation of Large Language Models（2021） | LoRA | [arXiv:2106.09685](https://arxiv.org/abs/2106.09685) | 19 |
| Training Compute-Optimal Large Language Models（2022, Chinchilla） | データとモデルの釣り合い | [arXiv:2203.15556](https://arxiv.org/abs/2203.15556) | 15 |
| Training language models to follow instructions with human feedback（2022, InstructGPT） | SFTとRLHF | [arXiv:2203.02155](https://arxiv.org/abs/2203.02155) | 18, 22 |
| ReAct（2022） | 考える＋行動するエージェント | [arXiv:2210.03629](https://arxiv.org/abs/2210.03629) | 23 |
| FlashAttention（2022） | 速いAttention | [arXiv:2205.14135](https://arxiv.org/abs/2205.14135) | 16 |
| Direct Preference Optimization（2023, DPO） | 報酬モデルなしの好みの学習 | [arXiv:2305.18290](https://arxiv.org/abs/2305.18290) | 22 |
| QLoRA（2023） | 4ビット＋LoRA | [arXiv:2305.14314](https://arxiv.org/abs/2305.14314) | 19 |
| RoFormer（2021, RoPE） | 回転位置埋め込み | [arXiv:2104.09864](https://arxiv.org/abs/2104.09864) | 25 |
| Retrieval-Augmented Generation（2020） | RAG | [arXiv:2005.11401](https://arxiv.org/abs/2005.11401) | 21 |

詳しい出典は[付録J 参考文献](https://sakunokyumei.github.io/llm-research-textbook/j-references.html)にまとめてあります。

#### 📝 確認問題（`ch26-q08` · Lv.3 · 論文の3回読みと「主張（Claim）」と「証拠（Evidence）」の切り分け）

新しいAI論文のタイトルと要旨（Abstract）に「我々の手法は従来法を劇的に上回る画期的性能を達成した」と書いてありました。批判的に論文を読む研究者が、本文の実験セクションで真っ先に確かめるべき「証拠」のチェック項目として最も重要なのはどれですか？

- **a.** ①ベースラインの公平性（パラメータ・学習量等）、②複数シードや信頼区間の差、③アブレーションでの寄与、④リークや限界の記述
- **b.** 論文の総ページ数が50ページ以上あるか、掲載されているグラフや図がカラフルで見栄えが良いかどうかだけを基準に信頼する
- **c.** 別のLLMに「この論文はすごいですか？」と3行で要約させて結論だけ読み、原文の図表・数式・実験設定は一度も確認しない
- **d.** 著者リストに有名な大企業やトップ大学の名前が入っているかだけを確認し、本文中の実験条件や数値の表には一切目を通さない

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：要旨（Abstract）の強い言葉（主張：Claim）をそのまま信じず、本文の実験結果（証拠：Evidence）で裏づける視点を思い出しましょう。
- **ヒント 2**：公平なベースライン比較・統計的ばらつき・アブレーション・限界記述の4観点を点検する選択肢を選びます。
- **正解**：`a`

**考え方**：論文の要旨は最大の売り込み文句が書かれる場所です。同じ論文を3回読み（1回目10分で全体像 → 2回目1時間で本文 → 3回目で数式とコード再現）しながら、「主張が本当に実験証拠で支えられているか？」を点検します。

1. 主張（Claim）を抽出する：「何を新しく提案し、何に勝ったと言っているか」。
1. 証拠（Evidence）を点検する：ベースラインの強さ、計算量・パラメータの一致、ばらつき、アブレーション。
1. 限界（Limitation）を読む：どんな条件なら成り立たないかも必ず確認する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q08

</details>

#### 📝 確認問題（`ch26-q09` · Lv.4 · 【発展】欠陥だらけの実験レポートから複数の問題点をすべて見抜く）

あるブログ記事に次のような実験報告が載っていました：
> 「自作の新Attentionを入れたモデル（パラメータ数200万・学習率調整済み）を1回だけ学習したら、デフォルト設定のGPT-2型（パラメータ数100万）より検証損失が0.02下がった！成功した生成例を3つ載せます。したがって新AttentionはあらゆるLLMで有効です。」

この記事の科学的な問題点として**すべて正しい指摘**になっている選択肢はどれですか？

- **a.** 唯一の欠点は成功した生成例が3つではなく100個載っていないことだけで、比較実験の設計と結論の妥当性は科学的に完璧である
- **b.** ①パラメータ2倍・調整労力も不公平、②1シードでばらつき不明、③成功例のみで失敗例・限界なし、④小規模実験からの過剰一般化
- **c.** ブログ記事の形式で公開していることが唯一の問題点なので、同じ実験結果のまま英語のPDF論文へ書き直せば科学的に正しくなる
- **d.** 検証損失が `0.02` しか下がっていないのが問題であり、提案モデルのパラメータをさらに10倍へ増やして `0.50` 下げれば公平な証明になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：引用されたブログ記事を「比較条件の公平さ」「乱数シードの回数」「例の選び方」「結論の射程」の4か所に分けて点検しましょう。
- **ヒント 2**：パラメータ数200万 vs 100万・1回だけ・成功例3つのみ・「あらゆるLLMで有効」という4つの赤信号をすべて指摘している選択肢を選びます。
- **正解**：`b`

**考え方**：SNSやブログの「すごい新手法！」投稿の多くは、この4点（不公平な条件・1シード・チェリーピッキング・過剰一般化）のどれかを含んでいます。これらを一瞬で見抜ける目こそ、この教科書で育てた研究者の眼です。

1. 条件の不公平：パラメータ2倍（200万 vs 100万）＋片方だけチューニング。
1. 統計の不備：1シードのみでは 0.02 差がノイズか実力か分からない。
1. 報告の偏り：成功例3つだけで失敗例・限界が書かれていない。
1. 過剰一般化：小規模1設定の結果から「あらゆるLLMで有効」と言い過ぎている。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q09

</details>

## 2030年への地図

あなたは2026年に入学して、2030年に卒業する予定です。この教科書を半年〜1年で終えたとすると、まだ3年以上あります。その時間を、どう使いましょう？ ひとつの例として、地図を描いてみます。

| 時期 | 技術 | 資格・実績 | 発信 |
|---|---|---|---|
| 2026年度（1年） | この教科書を完走。Python・Linux・Gitに慣れる | 基本情報技術者 | GitHubに教科書のコードと公開ページを置く |
| 2027年度（2年） | 論文を月1本読み、1本を再現する。Kaggleのコンペに初参加 | 応用情報技術者（AI関連の資格も興味があれば） | 学んだことを技術ブログに書く |
| 2028年度（3年） | インターンで実務を経験。OSS（transformersなど）の小さな修正に挑戦 | Kaggleでメダルを目指す | 勉強会やLT（短い発表）で話す |
| 2029年度（4年） | 卒業研究。小さくても、自分の問いに自分の実験で答える | — | 研究をまとめて公開（学会の学生向けの場や、論文として） |

> **✨ コツ：地図はあくまで例**
>
> この地図どおりに進む必要はまったくありません。寄り道や回り道が、一番の財産になることも多いのです。ただ、「いつか」ではなく「いつまでに」を決めておくと、人は意外と進めます。手帳かスマホのカレンダーに、最初の一歩（例：「来月、論文を1本読む」）だけでも書いておきましょう。

> **¥0 無料枠のコツ：0円で学び続けるために**
>
> - **論文**：arXivは無料。Hugging Faceの「Papers」ページでは、話題の論文がまとまって見られます
> - **計算**：Colab・Kaggleの無料GPU。大きな実験は、量子化とLoRAで小さくする（第19章）
> - **仲間**：Kaggleのディスカッション、勉強会（connpassなどで探せます）、学校の友だち
> - **教材**：Stanford CS336（LLMを作る講義、資料が公開されている）、Andrej Karpathyの「Neural Networks: Zero to Hero」（動画）、Hugging Face LLM Course など。この教科書を終えたあなたなら、英語の教材も十分に読めるはずです

**ミナト**：4年後の自分なんて想像もつかないけど……地図があると、ちょっと安心しますね。

**ボンゴ先生**：地図は、道に迷ったときに見るものだ。ずっとにらめっこしている必要はない。迷ったら「今どこにいて、次の一歩は何か」だけ確かめればいい。勾配降下法と同じさ。遠くの谷底は見えなくても、足元の傾きは分かる。

#### 📝 確認問題（`ch26-q10` · Lv.2 · 再現性（Reproducibility）のために毎回記録すべき項目）

半年後の自分や他の研究者がまったく同じ実験をやり直せる（**再現性**を保つ）ために、`experiment_001.json` のような記録ファイルに毎回残しておくべき情報として最も適切な組み合わせはどれですか？

- **a.** 実験を行った当日の天気や研究室の室温だけを詳しく記録し、学習率・ステップ数・乱数シード・ライブラリ版は一切残さない
- **b.** 複数回回したうち最も良かったシードの損失値1個だけを手書きメモへ残し、使ったコードや設定ファイルは毎回上書き消去する
- **c.** Hugging Faceの秘密アクセストークンやアカウントパスワードをそのまま実験ログJSONへ書き込み、Publicリポジトリで公開する
- **d.** 問い・データと分割法・ハイパーパラメータ・乱数シード一覧・Pythonや主要ライブラリのバージョン・全シードの実験結果

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：半年後の自分や第三者がまったく同じ実験を再現するために必要な「設計・設定・環境（版）・結果」の項目を整理しましょう。
- **ヒント 2**：本文の `record = {...}` で `platform.python_version()` や `importlib.metadata.version(...)` まで保存していた理由に注目してください。
- **正解**：`d`

**考え方**：ライブラリ（Gradioやtransformers、PyTorch）はバージョンが少し変わるだけでデフォルト挙動や数値が変わることがあるため、`importlib.metadata.version` で版まで自動記録しておくのがプロの習慣です。

1. `config`：モデル構造・学習率・ステップ数・データ分割・`seeds`。
1. `versions`：`python`, `torch`, `transformers` などの正確な版番号。
1. `results`：うまくいかなかった実験も含めてすべて残す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q10

</details>

## AIを作る人として

最後に、少しだけ真面目な話をさせてください。

> **💡 理論のポイント：AIを作る人の倫理**
>
> LLMは、とても強力な道具です。だからこそ、作る人には責任があります。
>
> - **データの権利**：学習に使う文章や画像には、書いた人・作った人がいます。ライセンスや利用規約を確かめ、権利を尊重する
> - **プライバシー**：個人情報を学習させない、漏らさない
> - **公平さ**：データの偏りは、モデルの偏りになる。特定の人たちに不利な出力をしないか、確かめる
> - **正直さ**：性能を盛らない。失敗も限界も書く（この章の報告書）
> - **安全**：悪用されたときに何が起きるかを考える。エージェントには最小限の権限を（第23章）
> - **環境**：大きな学習は大量の電気を使う。必要以上に大きくしない、小さく試してから大きく
>
> 技術は中立ではありません。それを使う人、作る人の判断で、良くも悪くもなります。この教科書で身につけた「自分の手で確かめる」力は、AIを**正しく疑い、正しく使う**力でもあります。

#### 📝 確認問題（`ch26-q11` · Lv.4 · 【総合・実験批評】卒業制作計画の3つの欠陥を診断し修正案を提案する）

後輩が次の卒業制作計画を相談してきました。
> 「『ボンゴ情報専門学校のFAQチャットボット』を作ります。学校のWeb記事と友だちのSNS投稿（許可なし）を集めてQwen-0.5BをLoRAでSFTします。同時にRAGも追加して、プロンプトも改良します。集めたFAQ 20問すべてで学習したあと、同じ20問を1回テストして、ベースモデルより良くなったら『LoRAのおかげで完璧になった！』とGitHub Pagesに全データ込みで公開します。」

この計画には **研究方法論・評価・倫理/公開** の面で深刻な欠陥が少なくとも4つ含まれています。それぞれの欠陥を指摘し、具体的な修正案を提示してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：後輩の計画を「①データ分割」「②変数の統制」「③統計的評価と誠実さ」「④倫理・権利・公開方式」の4つの視点で点検しましょう。
- **ヒント 2**：①同じ20問での訓練＆テスト（リーク）、②LoRA・RAG・プロンプトの同時変更（交絡）、③1回評価と過剰主張、④無断SNS収集と0.5BのGitHub Pages公開、の4点の欠陥と修正案を具体化します。
- **解答の目安**：① **データリーク（訓練とテストが同一）**：FAQ 20問すべてで学習して同じ20問でテストすると丸暗記を測るだけになる。修正：データを訓練・検証・独立テスト（または検索用文書と未知質問）に重複なく分離し、テスト質問は最終評価まで使わない。② **変数の交絡（LoRA＋RAG＋プロンプトを同時変更）**：3つを同時に変えると「LoRAのおかげ」と主張できない。修正：素のQwen → +プロンプト → +RAG → +LoRAのように1要素ずつアブレーション比較する。③ **評価の統計的不備と過剰主張**：20問1回だけでは偶然のばらつきが大きく「完璧」とは言えない。修正：正答率に加えてブートストラップ95%信頼区間や複数シード幅、失敗例・限界も明記する。④ **倫理・権利・公開方式の誤り**：友だちのSNS投稿を無断収集・公開するのはプライバシーと権利の侵害。またQwen-0.5Bはサイズ的にGitHub Pagesブラウザ公開に向かない。修正：権利処理済みの公開資料・自作データのみを使い、個人情報ゼロを確認し、0.5Bモデルのデモは一時GradioやOllama等で適切に運用する。

**考え方**：他人の実験計画の欠陥を的確に見つけて建設的な修正案を出せるようになれば、自分自身の研究でも同じ落とし穴を未然に防げるようになります。

1. データリーク指摘 → 訓練・検証・テストの厳密な分離へ修正。
1. 変数交絡指摘 → 1要素ずつ比べるアブレーション計画へ修正。
1. 統計・誠実さ指摘 → 信頼区間/複数シード・失敗例・限界の記載へ修正。
1. 倫理・技術指摘 → 無断データ削除・プライバシー配慮・モデル規模に合う公開方式へ修正。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q11

</details>

#### 📝 確認問題（`ch26-q12` · Lv.5 · 【卒業課題】自分の問い・仮説・実験計画・限界・倫理を1ページ設計する）

あなた自身の卒業制作の**研究計画書（ミニ版）**を設計してください。次の7項目をすべて具体的に書きましょう：
1. **研究の問い**（「〜は〜にどう影響するか？」の1文）
2. **反証可能な仮説**（予想と理由）
3. **ベースラインと変える変数**（変える条件は1つずつ）
4. **データ分割とデータリーク防止策**
5. **評価指標と統計チェック**（第20章の信頼区間や複数シード）
6. **再現性の記録項目**
7. **想定される限界と倫理的配慮**（権利・プライバシー・正直さ・安全・環境）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：本文のテーマA〜Fでも自作テーマでも構いません。「小さく回せて、客観的に測れて、実験で否定され得る（反証可能な）」形へ7項目を具体化しましょう。
- **ヒント 2**：1文の問い・反証可能仮説・1変数だけ変えるベースライン比較・データ分割とリーク防止・複数シード/信頼区間・再現性JSON・限界と倫理（権利・プライバシー・安全・正直さ）の7項目を漏れなく整理します。
- **解答の目安**：【例：テーマD（BPE語彙サイズのアブレーション）の場合】
1. **問い**：青空文庫の夏目漱石テキストで小規模GPTを学習するとき、BPEの語彙サイズ（文字単位 vs 500 vs 1500 vs 3000）は、同一ステップ・同一パラメータ規模における「1文字あたりの検証損失（BPC相当）」と生成速度にどう影響するか？
2. **仮説**：語彙サイズを文字単位から1500程度まで増やすと、1トークンで長い文脈を見られるため1文字あたり検証損失が下がり生成も速くなるが、3000以上に増やしすぎると希少トークンの学習回数が不足し検証損失が悪化する。
3. **ベースラインと変数**：文字単位（第13章）をベースラインとし、変える条件はBPE語彙サイズのみ。埋め込み層を除いた本体層数・幅・ステップ数・学習率・ドロップアウトは統一し、総パラメータ数も記録して比較する。
4. **データ分割とリーク防止**：作品単位（または先頭90%訓練／末尾10%検証・テスト）に先に分割してからBPEの学習とトークン化を行い、テストテキストの統計がBPE語彙作成に漏れないようにする。
5. **評価指標と統計**：語彙サイズが異なると1トークンあたり損失は直接比較できないため、全トークンの合計負対数尤度を「元テキストの文字数」で割った**1文字あたり損失**で統一比較する。各条件を3シードで実行し平均とばらつき幅を報告する。
6. **再現性**：設定・シード・`python`/`torch`版・乱数生成条件をJSONファイルに毎回保存し、GitHubにコードを公開する。
7. **限界と倫理**：小規模モデル・1作家の文体に限った結果であり大規模LLMへの一般化はできないと明記する。著作権保護期間満了の青空文庫のみを用い、失敗例や予想外の結果も隠さず報告し、無料T4で1時間以内に収まる省電力規模で実験する。

**考え方**：この7項目を自分の言葉で埋められた時点で、あなたは「AIを使う人」から「自分で問いを立て、検証し、誠実に報告できるAI研究者・開発者」への扉をすでに開けています！

1. 小さく回せて測れる問いを1文で書く。
1. 実験前に反証可能な仮説と、公平なベースライン・統制条件を決める。
1. データリーク防止・統計検証・再現性JSON・誠実な限界と倫理を計画に組み込む。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html#ex-ch26-q12

</details>

**✅ できたらチェック**

- [ ] 卒業制作のテーマを1つ決めた
- [ ] 問い・仮説・ベースライン・指標を、1ページに書いた
- [ ] 実験の設定と版を記録する仕組みを作った
- [ ] 最初の実験を回した（うまくいかなくてもOK）
- [ ] 論文を1本、3回読みの1回目だけでも読んだ
- [ ] 2030年までの地図に、最初の一歩を書き込んだ

> **📝 まとめ**
>
> - 卒業制作は「作る」＋「測る」。小さく、測れて、好きなテーマを
> - 研究は、問い → 仮説 → 実験計画 → 記録 → 結論（言えること・言えないこと）
> - **再現性**のために、設定・シード・版を記録する
> - 論文は3回に分けて読む。大事な論文はコードで再現する
> - AIを作る人には、データ・プライバシー・公平さ・正直さ・安全・環境への責任がある

**ボンゴ先生**：卒業おめでとう！ 🥁ドコドコドコドコ、ドン！

この教科書の最初に、私は「プールに飛び込もう」と言った。キミは飛び込んで、水を飲んで、息つぎを覚えて、今は自分の力で泳いでいる。ここから先の海は、どこまでも広い。でも、泳ぎ方はもう知っている。分からないことがあったら、小さく試して、確かめればいい。

やってみればわかる。いつだって、それが答えだ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html